In [1]:
# %pip install -q langchain-ollama

In [2]:
from langchain_ollama import ChatOllama
llm = ChatOllama(model="llama3.2:1b")  
llm.invoke("What is the capital of France?") # 프롬프트 : LLM호출 

AIMessage(content='The capital of France is Paris.', additional_kwargs={}, response_metadata={'model': 'llama3.2:1b', 'created_at': '2026-03-13T02:33:15.042725Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3177313000, 'load_duration': 3094316900, 'prompt_eval_count': 32, 'prompt_eval_duration': 15296300, 'eval_count': 8, 'eval_duration': 58651900, 'logprobs': None, 'model_name': 'llama3.2:1b', 'model_provider': 'ollama'}, id='lc_run--019ce50a-6db7-7ec3-9f54-e25524d0ac0f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 32, 'output_tokens': 8, 'total_tokens': 40})

In [3]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser  # output parser

prompt_template = PromptTemplate(
    template = "What is the capital of {country}?",    # Return the name of the city. (수도만 나오게끔 하려면)
    input_variables=["country"],
)

prompt = prompt_template.invoke({"country":"France"})  
ai_message = llm.invoke(prompt)
ai_message

AIMessage(content='The capital of France is Paris.', additional_kwargs={}, response_metadata={'model': 'llama3.2:1b', 'created_at': '2026-03-13T02:33:15.2252415Z', 'done': True, 'done_reason': 'stop', 'total_duration': 161505700, 'load_duration': 90068100, 'prompt_eval_count': 32, 'prompt_eval_duration': 9290000, 'eval_count': 8, 'eval_duration': 58531900, 'logprobs': None, 'model_name': 'llama3.2:1b', 'model_provider': 'ollama'}, id='lc_run--019ce50a-7a36-7961-be14-f8c8cfaffd8f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 32, 'output_tokens': 8, 'total_tokens': 40})

In [4]:
output_parser = StrOutputParser()  # output parser 정
answer = output_parser.invoke(llm.invoke(prompt))
answer

'The capital of France is Paris.'

In [5]:
# ai_message와 answer의 차이점

# ai_message는 AIMessage 클래스 (BaseMessage)로 나온다 
# asnwer 은 string형식으로 나온다

# base 보다는 string이 더 편리 -> 프론트로 전달한다고 가정 : 프론트에는 baseMessage가 없어 parsing불가 -> string으로..

In [6]:
from langchain_core.output_parsers import JsonOutputParser

country_detail_prompt = PromptTemplate(
    template = """Give following information about {country}:
    - Capital
    - Population
    - Language
    - Currency

    return it in JSON format. and return the JSON dictionary only
    """,
    input_variables=["country"],
)

country_detail_prompt = country_detail_prompt.invoke({"country" : "France"})

output_parser_json = JsonOutputParser()
output_parser_json.invoke(llm.invoke(country_detail_prompt))  # 에러 발생

{'capital': 'Paris',
 'population': 67.2,
 'language': 'French',
 'currency': 'Euro'}

In [7]:
country_detail_prompt

StringPromptValue(text='Give following information about France:\n    - Capital\n    - Population\n    - Language\n    - Currency\n\n    return it in JSON format. and return the JSON dictionary only\n    ')

In [8]:
error_json = llm.invoke(country_detail_prompt)
error_json.content

'```\n{\n    "capital": "Paris",\n    "population": 67.2 million,\n    "language": "French",\n    "currency": "Euro"\n}\n```'

In [9]:
# 사실상 error_json은 str 타입이다 -> str인데 JSONOutput을 활용해서 에러가 나는 것
# type(error_json.content)  
type(ai_message.content)  # str
# type(error_json.content)

str

In [10]:
# 가장 큰 문제 : 형식이 확실하지 않다.
# '```\n{\n    "capital": "Paris",\n      [정상 작동, 댁셔너리 반환]
# '```json\n{\n    "capital": "Paris",\n   [에러]
# json 이 있을 수도, 없을 수도 있다 -> 답변이 나올 수도, 안나올 수도 있음

# replace로 대체할 수는 있겠으나 -> 될지 안될지 모르는 상황 -> 그래서 사용 불가
# JSON을 파싱은 사용 불가임

- pydantic으로 모델 설정하기 - json문제점 대체
- langchain에서 JSON 형태의 반환을 강제하는 방법

In [11]:
from pydantic import BaseModel, Field

# 어떤 output을 원하는건지 지정해준다
class CountryDetail(BaseModel):
    capital: str = Field(description="Capital city of the country")
    population: int = Field(description="Population of the country")
    language: str = Field(description="Official language of the country")
    currency: str = Field(description="Currency used in the country")


structured_llm = llm.with_structured_output(CountryDetail)

In [14]:
from langchain_core.output_parsers import JsonOutputParser

country_detail_prompt = PromptTemplate(
    template = """Give following information about {country}:
    - Capital
    - Population
    - Language
    - Currency

    return it in JSON format. and return the JSON dictionary only
    """,
    input_variables=["country"],
)

country_detail_prompt = country_detail_prompt.invoke({"country" : "France"})

json_ai_message2 = structured_llm.invoke(country_detail_prompt)  # 에러 발생

In [16]:
print(json_ai_message2)
print(json_ai_message2.capital)
print(json_ai_message2.population)   #

capital='Paris' population=67 language='French' currency='Euro'
Paris
67


In [17]:
json_ai_message2.model_dump()   # 모델이기 때문에 dump를 해줄 수 있음 -> JSON이 됨

{'capital': 'Paris',
 'population': 67,
 'language': 'French',
 'currency': 'Euro'}

In [18]:
json_ai_message2.model_dump()['capital']   # 접근 가능, model을 활용하면 안정적인 답변을 받아낼 수 있다.

'Paris'